# Task 1: Keystroke Identification (Person A)
**Question:** given one typing sample of `.tie5Roanl` (31 timing features), which of the 51 users typed it?

**Contract (frozen in `split.txt`, keys `TASK1_TRAIN_SESSIONS` / `TASK1_TEST_SESSIONS`):** train = sessions 1-5 and 8 (300 reps/user), test = sessions 6-7 (100 reps/user). Tasks 2 and 3 keep the paper's sessions 1-4 / 5-8 protocol; Task 1 is not compared with the paper. The test sessions sit between training sessions, so training includes typing from after the test days (session 8). The sessions 1-4 / 5-8 and 1-6 / 7-8 runs are in `experiments/task1_split_comparison/`.
The test set is touched once per model. Hyperparameters are fixed up front, and for the deep models the epoch count is chosen on a validation slice of the *training* sessions (session 8), never on test.

Results are appended to `../results.csv` as `(model, task, metric, value)`.

Sections: 1 Setup · 2 EDA · 3 Features · 4 Classical models · 5 Deep models · 6 Ablations · 7 Error analysis · 8 Summary

In [1]:
# Setup: run this notebook from the repo's notebooks/ folder.
# metrics.py, split.txt and data/ are one level up, in the repo root.
# On Colab: upload or clone the repo, then %cd into <repo>/notebooks before running.
import os
assert os.path.exists(os.path.join("..", "metrics.py")), "Run this notebook from the repo's notebooks/ folder."

In [2]:
import os, sys, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)
from metrics import append_result, load_split   # shared contract file, never edited

CFG = load_split()
SEED = CFG["SEED"]
np.random.seed(SEED); random.seed(SEED)

DATA = os.environ.get("KD_DATA", os.path.join(ROOT, "data", "DSL-StrongPasswordData.csv"))
RESULTS_PATH = os.environ.get("KD_RESULTS", os.path.join(ROOT, "results.csv"))
TASK = "task1_identification"

# Re-running the notebook must not duplicate rows: drop only THIS task's old rows.
if os.path.exists(RESULTS_PATH):
    _r = pd.read_csv(RESULTS_PATH)
    _r[_r.task != TASK].to_csv(RESULTS_PATH, index=False)
print("split:", {k: CFG[k] for k in ("TASK1_TRAIN_SESSIONS", "TASK1_TEST_SESSIONS")}, "| seed", SEED)

split: {'TASK1_TRAIN_SESSIONS': [1, 2, 3, 4, 5, 8], 'TASK1_TEST_SESSIONS': [6, 7]} | seed 42


## 1. Load data and apply the frozen split

In [3]:
df = pd.read_csv(DATA)
feat_cols = [c for c in df.columns if c not in ("subject", "sessionIndex", "rep")]
assert len(feat_cols) == 31 and df[feat_cols].isna().sum().sum() == 0
assert df.subject.nunique() == 51 and (df.groupby("subject").size() == 400).all()

H_idx  = [i for i, c in enumerate(feat_cols) if c.startswith("H.")]
DD_idx = [i for i, c in enumerate(feat_cols) if c.startswith("DD.")]
UD_idx = [i for i, c in enumerate(feat_cols) if c.startswith("UD.")]
assert (len(H_idx), len(DD_idx), len(UD_idx)) == (11, 10, 10)

classes = sorted(df.subject.unique())
label = {s: i for i, s in enumerate(classes)}
train = df[df.sessionIndex.isin(CFG["TASK1_TRAIN_SESSIONS"])].reset_index(drop=True)
test  = df[df.sessionIndex.isin(CFG["TASK1_TEST_SESSIONS"])].reset_index(drop=True)
ytr, yte = train.subject.map(label).values, test.subject.map(label).values
print(df.shape, "| train", train.shape, "| test", test.shape)
print("train reps per user:", train.groupby("subject").size().unique(), "| test:", test.groupby("subject").size().unique())

(20400, 34) | train (15300, 34) | test (5100, 34)
train reps per user: [300] | test: [100]


## 2. EDA

In [4]:
# 2.1 Scale of each feature: DD/UD around the 5th and 6th keystrokes are far larger (the pause mid-password)
fig, ax = plt.subplots(figsize=(8, 7))
order = np.argsort(df[feat_cols].median().values)
ax.boxplot([df[feat_cols[i]].clip(upper=3) for i in order], vert=False, showfliers=False)
ax.set_yticklabels([feat_cols[i] for i in order])
ax.set_xlabel("seconds"); ax.set_title("Feature distributions (all data)")
plt.tight_layout(); plt.show()

# 2.2 Structural redundancy: DD = H(key1) + UD for every digraph
resid = np.abs(df[[feat_cols[i] for i in DD_idx]].values - df[[feat_cols[i] for i in UD_idx]].values - df[[feat_cols[i] for i in H_idx[:10]]].values)
print("max |DD - UD - H| =", resid.max().round(6))
rank = np.linalg.matrix_rank(df[feat_cols].values - df[feat_cols].values.mean(0))
print("numerical rank of the 31 features:", rank, "(tree models don't care; LR/Mahalanobis do)")
print("fraction of negative UD (key overlap):", (df[[feat_cols[i] for i in UD_idx]] < 0).values.mean().round(4))

max |DD - UD - H| = 0.0
numerical rank of the 31 features: 21 (tree models don't care; LR/Mahalanobis do)
fraction of negative UD (key overlap): 0.1084


In [5]:
# 2.3 The "fingerprint": mean profile of 6 users across the 31 features
rng = np.random.RandomState(SEED)
demo_users = rng.choice(classes, 6, replace=False)
prof = df[df.subject.isin(demo_users)].groupby("subject")[feat_cols].mean()
fig, ax = plt.subplots(figsize=(12, 4))
for u in demo_users:
    ax.plot(range(31), prof.loc[u].values, marker="o", ms=3, label=u)
ax.set_xticks(range(31)); ax.set_xticklabels(feat_cols, rotation=90, fontsize=7)
ax.set_yscale("log"); ax.set_ylabel("mean seconds (log)"); ax.legend(ncol=6, fontsize=8)
ax.set_title("Mean timing profile of 6 users"); plt.tight_layout(); plt.show()

# 2.4 Between-user vs within-user separation per feature (ANOVA F on train)
F, _ = f_classif(train[feat_cols].values, ytr)
fs = pd.Series(F, index=feat_cols).sort_values(ascending=False)
print("Most discriminative:\n", fs.head(8).round(1).to_string())
print("\nLeast discriminative:\n", fs.tail(5).round(1).to_string())

Most discriminative:
 H.Shift.r    963.1
H.t          850.8
H.i          601.6
H.l          509.0
H.period     502.2
H.n          476.7
H.o          466.3
H.e          465.9

Least discriminative:
 DD.e.five    119.8
DD.t.i       119.0
UD.e.five    118.0
UD.i.e        45.6
DD.i.e        41.8


In [6]:
# 2.5 Between-user vs within-user variation, and session drift
DDc = [feat_cols[i] for i in DD_idx]
df["total_time"] = df[DDc].sum(axis=1) + df["H.Return"]      # first press to last release

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
um = df.groupby("subject").total_time.agg(["mean", "std"]).sort_values("mean")
axes[0].errorbar(range(51), um["mean"], yerr=um["std"], fmt="o", ms=3, lw=0.8)
axes[0].set_title("Total typing time per user (mean ± std)"); axes[0].set_xlabel("users, sorted"); axes[0].set_ylabel("s")

sm = df.groupby(["subject", "sessionIndex"]).total_time.mean().unstack()
sm.T.plot(ax=axes[1], legend=False, alpha=0.25, color="gray")
sm.mean().plot(ax=axes[1], color="C3", lw=2.5)
axes[1].set_title("Per-user mean typing time by session"); axes[1].set_xlabel("session")

rel = sm.div(sm[1], axis=0)
axes[2].boxplot([rel[s] for s in range(1, 9)]); axes[2].set_xticklabels(range(1, 9)); axes[2].axhline(1, c="k", lw=0.8)
axes[2].set_title("Typing time relative to own session 1"); axes[2].set_xlabel("session")
plt.tight_layout(); plt.show()
print("Mean speed-up vs session 1, test sessions", CFG["TASK1_TEST_SESSIONS"], ":", (rel[CFG["TASK1_TEST_SESSIONS"]].mean().values - 1).round(3))

Mean speed-up vs session 1, test sessions [6, 7] : [-0.25 -0.27]


**What the EDA tells us:**
- **Users differ a lot.** Mean total typing time ranges from about 1.5 s to 5.4 s per user. Fast typists are also consistent (small error bars), while slow typists vary much more, so slow users are the likeliest to be confused with each other.
- **Hold times carry the most signal.** By ANOVA F-score the top 8 features are all hold times (`H.t`, `H.Shift.r`, `H.i`, ...). Hold times are short (about 0.05-0.12 s) and stable within a user. The between-key times around `i -> e` are the least discriminative.
- **The longest gaps** are `five -> Shift.r` and `e -> five`: moving to the number row and reaching for Shift to type the capital R.
- **Key overlap is common.** 10.8% of up-down times are negative (the next key is pressed before the last one is released).
- **Features are redundant.** DD = H + UD exactly, so the 31 features have rank 21. Tree models don't care, but linear and distance-based models do.
- **Typing drifts.** Mean typing time falls from about 3.2 s in session 1 to about 2.3 s in session 8; users are about 25-27% faster in the test sessions (6-7) than in session 1. Task 1 trains on sessions on both sides of the test sessions (1-5 and 8), so the model sees typing from before and after the test days; accuracy by test session is in section 7.

## 3. Features

In [7]:
def add_derived(d):
    X = d[feat_cols].copy()
    Hc  = [feat_cols[i] for i in H_idx]
    DDc_ = [feat_cols[i] for i in DD_idx]
    UDc = [feat_cols[i] for i in UD_idx]
    X["total_time"] = d[DDc_].sum(axis=1) + d["H.Return"]
    X["speed_keys_per_s"] = 11.0 / X["total_time"]
    for n, cols in (("H", Hc), ("DD", DDc_), ("UD", UDc)):
        X[f"mean_{n}"] = d[cols].mean(axis=1)
        X[f"std_{n}"]  = d[cols].std(axis=1)
    X["n_overlaps"] = (d[UDc] < 0).sum(axis=1)
    X["ratio_H_DD"] = X["mean_H"] / X["mean_DD"]
    return X

# Main experiments use the 31 raw features. Global scaler is fit on TRAIN ONLY.
Xtr_raw, Xte_raw = train[feat_cols].values, test[feat_cols].values
scaler = StandardScaler().fit(Xtr_raw)
Xtr, Xte = scaler.transform(Xtr_raw), scaler.transform(Xte_raw)

def to_seq(X):
    # (N, 11 keystrokes, 3 channels): hold, down-down to next key, up-down to next key; last key has no next -> 0
    S = np.zeros((len(X), 11, 3), dtype=np.float32)
    S[:, :, 0] = X[:, H_idx]; S[:, :10, 1] = X[:, DD_idx]; S[:, :10, 2] = X[:, UD_idx]
    return S
Str, Ste = to_seq(Xtr), to_seq(Xte)
print("tabular", Xtr.shape, "| sequence", Str.shape)

tabular (15300, 31) | sequence (15300, 11, 3)


## 4. Classical models

In [8]:
RESULTS, MODELS = {}, {}

def report(name, y_pred, extra=""):
    acc = accuracy_score(yte, y_pred); f1 = f1_score(yte, y_pred, average="macro")
    RESULTS[name] = dict(accuracy=acc, macro_f1=f1, pred=np.asarray(y_pred))
    append_result(name, TASK, "accuracy", acc, path=RESULTS_PATH)
    append_result(name, TASK, "macro_f1", f1, path=RESULTS_PATH)
    print(f"{name:<24} acc={acc:.4f}  macroF1={f1:.4f}  {extra}")

def run_sk(name, model, Xa, Xb):
    t = time.time(); model.fit(Xa, ytr); MODELS[name] = model
    report(name, model.predict(Xb), f"({time.time() - t:.0f}s)")

run_sk("LogisticRegression", LogisticRegression(C=10, max_iter=3000), Xtr, Xte)
run_sk("SVM-RBF", SVC(C=10, kernel="rbf", gamma="scale"), Xtr, Xte)
run_sk("RandomForest", RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED), Xtr, Xte)

LogisticRegression       acc=0.8157  macroF1=0.8116  (0s)


SVM-RBF                  acc=0.8792  macroF1=0.8777  (3s)


RandomForest             acc=0.9286  macroF1=0.9273  (3s)


In [9]:
from xgboost import XGBClassifier
run_sk("XGBoost", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=SEED), Xtr, Xte)

XGBoost                  acc=0.9324  macroF1=0.9317  (10s)


## 5. Deep models
Epoch count is picked on session 8 (held out of an inner training run on sessions 1-5), then the model is retrained on all six training sessions (1-5 and 8) for that many epochs and scored once on test.

In [10]:
import torch, torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"
N_CLS = len(classes)
print("device:", device)

def predict(m, X):
    m.eval()
    with torch.no_grad():
        out = [m(torch.as_tensor(X[i:i + 2048], dtype=torch.float32, device=device)).argmax(1).cpu() for i in range(0, len(X), 2048)]
    return torch.cat(out).numpy()

def train_net(model_fn, X, y, epochs, Xv=None, yv=None, lr=2e-3, wd=1e-3, bs=128):
    torch.manual_seed(SEED)
    m = model_fn().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.CrossEntropyLoss(label_smoothing=0.05)
    Xt = torch.as_tensor(X, dtype=torch.float32, device=device); yt = torch.as_tensor(y, dtype=torch.long, device=device)
    hist = []
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            if len(idx) < 2: continue                      # BatchNorm needs >1 sample
            opt.zero_grad(); lossf(m(Xt[idx]), yt[idx]).backward(); opt.step()
        if Xv is not None:
            hist.append(accuracy_score(yv, predict(m, Xv)))
    return m, hist

def run_dl(name, model_fn, Xa, Xb, max_epochs=80, **kw):
    t = time.time()
    sess = train.sessionIndex.values
    val = sess == max(CFG["TASK1_TRAIN_SESSIONS"])
    _, hist = train_net(model_fn, Xa[~val], ytr[~val], max_epochs, Xa[val], ytr[val], **kw)
    best = int(np.argmax(hist)) + 1
    m, _ = train_net(model_fn, Xa, ytr, best, **kw)
    MODELS[name] = m
    report(name, predict(m, Xb), f"(epochs={best}, val_acc={max(hist):.3f}, {time.time() - t:.0f}s)")
    return hist

class MLP(nn.Module):
    def __init__(self, d=31, h=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, N_CLS))
    def forward(self, x): return self.net(x)

class CNN1D(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(3, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128 * 11, 256), nn.ReLU(), nn.Dropout(p), nn.Linear(256, N_CLS))
    def forward(self, x): return self.head(self.conv(x.transpose(1, 2)))

class BiLSTM(nn.Module):
    def __init__(self, h=128, p=0.3):
        super().__init__()
        self.lstm = nn.LSTM(3, h, num_layers=2, batch_first=True, bidirectional=True, dropout=p)
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(4 * h, N_CLS))
    def forward(self, x):
        o, _ = self.lstm(x)
        return self.head(torch.cat([o.mean(1), o.max(1).values], 1))

device: cpu


In [11]:
hist_mlp  = run_dl("MLP", MLP, Xtr, Xte)
hist_cnn  = run_dl("1D-CNN", CNN1D, Str, Ste)
hist_lstm = run_dl("BiLSTM", BiLSTM, Str, Ste)

plt.figure(figsize=(6, 3.5))
for n, h in (("MLP", hist_mlp), ("1D-CNN", hist_cnn), ("BiLSTM", hist_lstm)):
    plt.plot(range(1, len(h) + 1), h, label=n)
plt.xlabel("epoch"); plt.ylabel("val acc (session 8)"); plt.legend(); plt.title("Epoch selection on held-out training session")
plt.tight_layout(); plt.show()

MLP                      acc=0.9386  macroF1=0.9375  (epochs=73, val_acc=0.850, 16s)


1D-CNN                   acc=0.9467  macroF1=0.9455  (epochs=62, val_acc=0.857, 152s)


BiLSTM                   acc=0.9088  macroF1=0.9070  (epochs=58, val_acc=0.775, 275s)


## 6. Ablations
Does hand-crafted feature engineering help, and does the answer depend on the model family?

In [12]:
Xtr_d, Xte_d = add_derived(train).values, add_derived(test).values
sc_d = StandardScaler().fit(Xtr_d)
run_sk("LogReg (+derived)", LogisticRegression(C=10, max_iter=3000), sc_d.transform(Xtr_d), sc_d.transform(Xte_d))
run_sk("RandomForest (+derived)", RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED), Xtr_d, Xte_d)

# Training-data size: accuracy of RandomForest vs. number of training reps per user (first n reps of the training sessions)
rows = []
for n in (10, 25, 50, 100, 200, 300):
    keep = train.groupby("subject").cumcount().values < n
    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(Xtr[keep], ytr[keep])
    rows.append((n, accuracy_score(yte, m.predict(Xte))))
lc = pd.DataFrame(rows, columns=["reps_per_user", "accuracy"]); print(lc.round(4).to_string(index=False))
plt.figure(figsize=(5, 3.2)); plt.plot(lc.reps_per_user, lc.accuracy, "o-"); plt.xscale("log")
plt.xlabel("training reps per user"); plt.ylabel("test accuracy"); plt.title("RandomForest data-efficiency"); plt.tight_layout(); plt.show()

LogReg (+derived)        acc=0.8373  macroF1=0.8341  (1s)


RandomForest (+derived)  acc=0.9216  macroF1=0.9198  (5s)


 reps_per_user  accuracy
            10    0.2631
            25    0.4080
            50    0.5180
           100    0.6441
           200    0.8190
           300    0.9255


## 7. Error analysis (best model)

In [13]:
best_name = max(RESULTS, key=lambda k: RESULTS[k]["accuracy"])
bp = RESULTS[best_name]["pred"]
print("best model:", best_name, round(RESULTS[best_name]["accuracy"], 4))

# accuracy by test session (drift check)
print("\nAccuracy by test session:")
print(pd.Series(bp == yte).groupby(test.sessionIndex.values).mean().round(4).to_string())

# per-user recall: hardest users, and most confused pairs
cm = confusion_matrix(yte, bp)
rec = pd.Series(np.diag(cm) / cm.sum(1), index=classes).sort_values()
print("\nHardest users (recall):\n", rec.head(8).round(3).to_string())
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], classes[i], classes[j]) for i in range(51) for j in range(51) if off[i, j] > 0), reverse=True)[:8]
print("\nMost confused (count, true -> predicted):")
for c, a, b in pairs: print(f"  {c:3d}  {a} -> {b}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(cm, cmap="Blues", ax=axes[0], cbar=False, xticklabels=False, yticklabels=False)
axes[0].set_title(f"Confusion matrix: {best_name}"); axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")
rec.plot.bar(ax=axes[1], width=0.8); axes[1].set_xticks([]); axes[1].set_ylabel("recall"); axes[1].set_title("Per-user recall (sorted)")
plt.tight_layout(); plt.show()

# what the forest uses
imp = pd.Series(MODELS["RandomForest"].feature_importances_, index=feat_cols).sort_values()
imp.tail(12).plot.barh(figsize=(5, 4), title="RandomForest top-12 feature importances"); plt.tight_layout(); plt.show()

best model: 1D-CNN 0.9467

Accuracy by test session:
6    0.9494
7    0.9439

Hardest users (recall):
 32    0.50
21    0.71
37    0.83
8     0.83
54    0.84
2     0.88
47    0.88
34    0.89

Most confused (count, true -> predicted):
   15  32 -> 3
   13  8 -> 11
   11  54 -> 20
   11  21 -> 32
    9  21 -> 3
    8  32 -> 12
    5  51 -> 57
    5  32 -> 37


## 8. Summary and saved artifacts

In [14]:
import joblib
res = pd.DataFrame({k: {"accuracy": v["accuracy"], "macro_f1": v["macro_f1"]} for k, v in RESULTS.items()}).T.sort_values("accuracy", ascending=False)
display(res.round(4))

plt.figure(figsize=(7, 4))
res.accuracy.iloc[::-1].plot.barh(color="C0"); plt.xlim(max(0, res.accuracy.min() - 0.05), 1.0)
plt.xlabel("test accuracy (sessions 6-7)"); plt.title("Task 1: 51-way identification"); plt.tight_layout(); plt.show()

# Save the best sklearn/XGBoost model for the Streamlit tab (trained on sessions 1-5 and 8).
classical = [k for k in ("LogisticRegression", "SVM-RBF", "RandomForest", "XGBoost") if k in RESULTS]
bc = max(classical, key=lambda k: RESULTS[k]["accuracy"])
os.makedirs(os.path.join(ROOT, "models"), exist_ok=True)
joblib.dump({"name": bc, "model": MODELS[bc], "scaler": scaler, "classes": classes, "feat_cols": feat_cols},
            os.path.join(ROOT, "models", "task1_best_classical.joblib"))
print("saved", bc, "->", os.path.join(ROOT, "models", "task1_best_classical.joblib"))
print("results rows for this task:", (pd.read_csv(RESULTS_PATH).task == TASK).sum())

,accuracy,macro_f1
1D-CNN,0.9467,0.9455
MLP,0.9386,0.9375
XGBoost,0.9324,0.9317
RandomForest,0.9286,0.9273
RandomForest (+derived),0.9216,0.9198
BiLSTM,0.9088,0.9070
SVM-RBF,0.8792,0.8777
LogReg (+derived),0.8373,0.8341
LogisticRegression,0.8157,0.8116


saved XGBoost -> /Users/Shikhar/Downloads/keystroke-dynamics-auth/models/task1_best_classical.joblib
results rows for this task: 18
